In [1]:
import os
import numpy as np
import pandas as pd
import librosa

# =========================
# CONFIG
# =========================
INPUT_WAV_DIR = r"C:\Users\66617\Desktop\self collected control dataSet_WAV"
OUTPUT_CSV_DIR = r"C:\Users\66617\Desktop\phase2_fft_csv"

SAMPLE_RATE = 16000
FFT_WINDOW_SEC = 0.5   # 500 ms
FFT_HOP_SEC = 0.25     # 250 ms
TARGET_FREQ_BINS = 128
TARGET_TIME_FRAMES = 120

os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)

# =========================
# SINGLE FILE -> FFT CSV
# =========================
def extract_fft_feature_from_wav(
    wav_path: str,
    sr: int = SAMPLE_RATE,
    win_sec: float = FFT_WINDOW_SEC,
    hop_sec: float = FFT_HOP_SEC,
    n_bins: int = TARGET_FREQ_BINS,
    target_frames: int = TARGET_TIME_FRAMES,
    center: bool = True,
    use_log: bool = False,
    debug: bool = False
) -> np.ndarray:
    """
    Extract FFT spectrogram-like feature from WAV and return shape (128, 120)

    หมายเหตุ:
    - use_log=False -> ใช้ magnitude ตรง ๆ
    - use_log=True  -> ใช้ log1p(magnitude)
    - center=True/False ควรลองให้ตรงกับ representation ที่ใกล้ Phase 1 มากที่สุด
    """

    y, _ = librosa.load(wav_path, sr=sr, mono=True)

    n_fft = int(win_sec * sr)      # 8000
    hop_length = int(hop_sec * sr) # 4000

    stft = librosa.stft(
        y,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        center=center
    )

    mag = np.abs(stft)

    if use_log:
        feat = np.log1p(mag)
    else:
        feat = mag

    # keep first 128 bins
    feat = feat[:n_bins, :]

    # make time dimension = 120
    current_frames = feat.shape[1]
    if current_frames != target_frames:
        x_old = np.linspace(0, 1, current_frames)
        x_new = np.linspace(0, 1, target_frames)

        feat_resized = np.zeros((n_bins, target_frames), dtype=np.float32)
        for i in range(n_bins):
            feat_resized[i] = np.interp(x_new, x_old, feat[i])

        feat = feat_resized

    feat = feat.astype(np.float32)

    if debug:
        print(f"\n[DEBUG] {os.path.basename(wav_path)}")
        print("shape    :", feat.shape)
        print("min/max  :", feat.min(), feat.max())
        print("mean/std :", feat.mean(), feat.std())

    return feat


def save_feature_to_csv(feature: np.ndarray, csv_path: str):
    """
    Save (128,120) feature to CSV without header/index
    """
    pd.DataFrame(feature).to_csv(csv_path, header=False, index=False)


# =========================
# BATCH PROCESS
# =========================
wav_files = [f for f in os.listdir(INPUT_WAV_DIR) if f.lower().endswith(".wav")]
wav_files.sort()

print(f"[INFO] Found {len(wav_files)} wav files")

summary_rows = []

for filename in wav_files:
    wav_path = os.path.join(INPUT_WAV_DIR, filename)
    csv_name = os.path.splitext(filename)[0] + ".csv"
    csv_path = os.path.join(OUTPUT_CSV_DIR, csv_name)

    feat = extract_fft_feature_from_wav(
        wav_path,
        sr=SAMPLE_RATE,
        win_sec=FFT_WINDOW_SEC,
        hop_sec=FFT_HOP_SEC,
        n_bins=TARGET_FREQ_BINS,
        target_frames=TARGET_TIME_FRAMES,
        center=True,     # ลอง True ก่อน
        use_log=False,   # ลอง magnitude ก่อน
        debug=False
    )

    save_feature_to_csv(feat, csv_path)

    summary_rows.append({
        "file": filename,
        "csv_file": csv_name,
        "shape": str(feat.shape),
        "min": float(feat.min()),
        "max": float(feat.max()),
        "mean": float(feat.mean()),
        "std": float(feat.std())
    })

    print(f"[OK] Saved: {csv_name}")

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_CSV_DIR, "fft_csv_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n[INFO] Done")
print("[INFO] Output CSV folder:", OUTPUT_CSV_DIR)
print("[INFO] Summary file     :", summary_path)

[INFO] Found 48 wav files
[OK] Saved: SC01_AloneBaseline_L0.csv
[OK] Saved: SC02_FaceToFaceConversation_L2.csv
[OK] Saved: SC03_OtherSpeech_L1.csv
[OK] Saved: SC04_GroupConversation_L2.csv
[OK] Saved: SC05_AloneMusic_L0.csv
[OK] Saved: SC06_AlonePodcast_L0.csv
[OK] Saved: SC07_AloneOnTheTrain_L0.csv
[OK] Saved: SC08_AloneWalking_L0.csv
[OK] Saved: SC09_AlnoeWalking_L0.csv
[OK] Saved: SC10_OtherSpeech_L1.csv
[OK] Saved: SC11_TalkingWhileWalking_L2.csv
[OK] Saved: SC12_TalkingWhileEating_L2.csv
[OK] Saved: SC13_TalkingWhileWalking_L2.csv
[OK] Saved: SC14_FaceToFaceConversationOffice_L2.csv
[OK] Saved: SC15_OtherSpeech_L1.csv
[OK] Saved: SC16_FaceToFaceConversation_L2.csv
[OK] Saved: SC17_FaceToFaceConversation_L2.csv
[OK] Saved: SC18_OhterSpeech_L1.csv
[OK] Saved: SC19_FaceToFaceConversation_L2.csv
[OK] Saved: SC20_FaceToFaceConversation_L2.csv
[OK] Saved: SC21_OtherSpeech_L1.csv
[OK] Saved: SC22_FaceToFaceCoversation_L2.csv
[OK] Saved: SC23_AloneWalking_L0.csv
[OK] Saved: SC24_AloneOnTh

In [4]:
import pandas as pd
import numpy as np

phase1_csv = r"C:\Users\66617\Desktop\audio_controlled_dataset\audio_imu_conversational_data\audio_features\p11\feat_30sec\group_study_59_0.csv"
phase2_csv = r"C:\Users\66617\Desktop\phase2_fft_csv\SC01_AloneBaseline_L0.csv"

a = pd.read_csv(phase1_csv, header=None).values.astype(np.float32)
b = pd.read_csv(phase2_csv, header=None).values.astype(np.float32)

print("Phase1 shape    :", a.shape)
print("Phase1 min/max  :", a.min(), a.max())
print("Phase1 mean/std :", a.mean(), a.std())

print("\nPhase2 shape    :", b.shape)
print("Phase2 min/max  :", b.min(), b.max())
print("Phase2 mean/std :", b.mean(), b.std())

Phase1 shape    : (128, 120)
Phase1 min/max  : 7.5581186e-10 7.903661e-06
Phase1 mean/std : 6.6301936e-08 2.869731e-07

Phase2 shape    : (128, 120)
Phase2 min/max  : 1.9100944e-05 12.952003
Phase2 mean/std : 0.1475152 0.5505852
